# Dinka (Southwestern) topic classifier: live demo notebook
Loads the fine-tuned NLLB-encoder model from the Hugging Face Hub (`slickman-tyrus/dinka-sib200-classifier`) and serves it with Gradio.

**How to run:** `Runtime > Run all`. A GPU is optional (`Runtime > Change runtime type > T4 GPU` makes predictions faster). The last cell prints a **public link** (`https://....gradio.live`). Keep this notebook running while the link is in use; the link stops working when the session ends.

Why a notebook and not a hosted Space: current Hugging Face policy requires a paid plan for CPU-based Gradio Spaces, so this notebook is the free, reproducible way to run the app.

In [1]:
!pip -q install gradio sentencepiece huggingface_hub

In [2]:
import sys, torch
from huggingface_hub import snapshot_download

MODEL_REPO = "slickman-tyrus/dinka-sib200-classifier"
path = snapshot_download(MODEL_REPO)
sys.path.insert(0, path)
from model_def import load_predictor

predict = load_predictor(path)
print("model loaded on", "GPU" if torch.cuda.is_available() else "CPU")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

model loaded on GPU


In [3]:
EXAMPLES = [
    "Pan cɔl Turkey aci piu war adekdiet guɔ̈l piny ɣon kadiakic, Aegean tɔ ciem, Black sea tɔ ye nhom tueŋ ku Meditreanean tɔ ye cok piiny.",
    "Dek mɔ̈u amäth. Mɔ̈u akɔ̈c thöŋ gup tɛ̈ loi thok, ku ŋïcŋïc adan ye dek ake riil tör",
]
for t in EXAMPLES:
    p = predict([t])[0]
    print(sorted(p.items(), key=lambda kv: -kv[1])[:3], "\n  ", t[:80])

[('geography', 0.9822288751602173), ('travel', 0.013647916726768017), ('entertainment', 0.002122586825862527)] 
   Pan cɔl Turkey aci piu war adekdiet guɔ̈l piny ɣon kadiakic, Aegean tɔ ciem, Bla
[('health', 0.9590653777122498), ('science/technology', 0.013750585727393627), ('travel', 0.012029540725052357)] 
   Dek mɔ̈u amäth. Mɔ̈u akɔ̈c thöŋ gup tɛ̈ loi thok, ku ŋïcŋïc adan ye dek ake riil


In [4]:
import gradio as gr

def classify(text):
    text = (text or "").strip()
    if not text:
        return {}
    return predict([text])[0]

demo = gr.Interface(
    fn=classify,
    inputs=gr.Textbox(lines=4, label="Dinka sentence (Southwestern Dinka)"),
    outputs=gr.Label(num_top_classes=7, label="Predicted topic"),
    examples=[[e] for e in EXAMPLES],
    title="Dinka topic classifier",
    description=("Classifies a Dinka sentence into one of 7 topics (science/technology, travel, politics, sports, health, "
                 "entertainment, geography). Trained on SIB-200 (Southwestern Dinka, 701 translated Flores-200 sentences). "
                 "Limitations: small training set; other dialects such as Bor may be classified less reliably; "
                 "single sentences of news/travel-style text only."),
)
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://<random-id>.gradio.live (temporary, changes on every run)

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
